In [2]:
# Cell 1: Check whether the server and model survived
!ollama list

NAME                       ID              SIZE      MODIFIED    
nomic-embed-text:latest    0a109f422b47    274 MB    3 hours ago    
gemma4:e2b                 b37049369adf    4.6 GB    3 hours ago    


In [3]:
# Cell 2: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

/bin/bash: line 1: nvidia-smi: command not found


In [4]:
# Cell 3: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
pciutils is already the newest version (1:3.10.0-2build1).
lshw is already the newest version (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [5]:
# Cell 4: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
# Cell 5: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

In [7]:
# Cell 6: Install the Ollama Python client
!pip install -q ollama

In [8]:
# Cell 7: Download the text model
import ollama

print(ollama.pull("gemma4:e2b").status)

success


In [9]:
# Cell 8: Download the embedding model
import ollama

print(ollama.pull("nomic-embed-text").status)

success


In [10]:
# Cell 9: Turn one sentence into a vector

response = ollama.embed(
    model = "nomic-embed-text",
    input = "Cable glands seal the point where a cable enters an enclosure."
)

vector = response.embeddings[0]
print(len(vector))
print(vector[:5])

768
[0.030723225, 0.053521626, -0.14588973, -0.055553254, 0.024468042]


In [12]:
# Cell 10: Compare two sentences with cosine similarity

import math 
sentences = [
    "Cable glands seal the point where a cable enters an enclosure.",
    "How do I close the gap where a wire goes into a box?",
    "A busbar carries current between circuit breakers in a panel.",
]

# One call can embed several sentences; the vectors come back in the same order
vectors = ollama.embed(model = "nomic-embed-text", input = sentences).embeddings


def cosine (a, b):
    """Return the cosine similarity of two vectors (1 = same direction)."""
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(y * y for y in b))
    return dot / (length_a * length_b)

print("0 vs 1:", round(cosine(vectors[0], vectors[1]), 3))
print("0 vs 2:", round(cosine(vectors[0], vectors[2]), 3))
print("1 vs 2:", round(cosine(vectors[1], vectors[2]), 3))

0 vs 1: 0.574
0 vs 2: 0.495
1 vs 2: 0.532


In [13]:
# Cell 11: Cosine similarity with prints, on three-number vectors
import math


def cosine_verbose(a, b):
    """Return the cosine similarity of two vectors, printing each step."""
    print("a =", a)
    print("b =", b)

    # Pair the numbers: first with first, second with second, and so on
    pairs = list(zip(a, b))
    print("pairs (x, y) =", pairs)

    # Multiply each pair, then add the products
    products = [x * y for x, y in pairs]
    dot = sum(products)
    print("products x*y =", products)
    print("dot =", dot)

    # The size of each vector: square every number, add, take the square root
    squares_a = [x * x for x in a]
    squares_b = [y * y for y in b]
    length_a = math.sqrt(sum(squares_a))
    length_b = math.sqrt(sum(squares_b))
    print("squares of a =", squares_a, "-> length_a =", round(length_a, 3))
    print("squares of b =", squares_b, "-> length_b =", round(length_b, 3))

    # Divide by both sizes so only the direction counts
    score = dot / (length_a * length_b)
    print("score = dot / (length_a * length_b) =", round(score, 3))
    return score


cosine_verbose([1, 2, 3], [3, 2, 1])

a = [1, 2, 3]
b = [3, 2, 1]
pairs (x, y) = [(1, 3), (2, 2), (3, 1)]
products x*y = [3, 4, 3]
dot = 10
squares of a = [1, 4, 9] -> length_a = 3.742
squares of b = [9, 4, 1] -> length_b = 3.742
score = dot / (length_a * length_b) = 0.714


0.7142857142857143